Downloading and verifying the Dataset:

In [ ]:
import kagglehub
kagglehub.dataset_download('sadmanhsakib/hc18-preprocessed-dataset')

import os
print(os.listdir("/kaggle/input/datasets/sadmanhsakib/hc18-preprocessed-dataset/"))

Binary Mask Value Remapping:

In [ ]:
from fastai.vision.all import *
import numpy as np
from PIL import Image

class FetalMask(PILMask):
    """Custom PILMask subclass to map mask values of 255 down to class index 1."""
    @classmethod
    def create(cls, fn):
        img = PILImage.create(fn)
        arr = np.array(img)
        arr = (arr > 127).astype(np.uint8)
        return cls(Image.fromarray(arr))

# Proper TransformBlock that wires FetalMask.create into the pipeline
def FetalMaskBlock(codes=None):
    return TransformBlock(
        type_tfms=FetalMask.create,
        item_tfms=AddMaskCodes(codes=ifnone(codes, [str(i) for i in range(10)])),
        batch_tfms=IntToFloatTensor
    )

Calculating the Dice + Focal Combined Loss:
Focal loss focuses on hard-to-classify pixels/boundaries and Dice Loss Directly Optimizes the target metric. Combining both is crucial for medical image segmentation. 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
        
    def forward(self, logits, targets):
        # Softmax over class dimensions to get probabilities
        probs = F.softmax(logits, dim=1)
        # Target class 1 (fetal head)
        pred = probs[:, 1]
        target = (targets == 1).float()
        
        intersection = (pred * target).sum(dim=(1, 2))
        union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
        
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, logits, targets):
        # Cross entropy loss (reduction='none' to scale manually)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1.0 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

class DiceFocalLoss(nn.Module):
    """Combined Dice and Focal Loss for robust boundary segmentation."""
    def __init__(self, alpha=0.25, gamma=2.0, smooth=1e-6):
        super().__init__()
        self.dice = DiceLoss(smooth)
        self.focal = FocalLoss(alpha, gamma)
        
    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.focal(logits, targets)

The FastAI DataBlock & Dataloaders: To ensure training matches the exact train/validation splits used for YOLOv8.

In [ ]:
def parent_folder_splitter(items):
    train_idx = [i for i, o in enumerate(items) if 'train' in o.parts]
    val_idx   = [i for i, o in enumerate(items) if 'val'   in o.parts]
    return train_idx, val_idx

def get_mask_path(img_path):
    return Path(str(img_path).replace("images", "masks"))

FASTAI_DIR = Path("/kaggle/input/datasets/sadmanhsakib/hc18-preprocessed-dataset/fastai/")

fetal_block = DataBlock(
    blocks=(ImageBlock, FetalMaskBlock(codes=['background', 'fetal_head'])),
    get_items=get_image_files,
    get_y=get_mask_path,
    splitter=parent_folder_splitter,
    item_tfms=Resize(256, method=ResizeMethod.Squish),
    batch_tfms=aug_transforms(
        mult=1.0,
        do_flip=True,
        flip_vert=False, # fetal head has anatomical up/down orientation
        max_rotate=15.0,
        max_zoom=1.15,
        max_lighting=0.15,
        max_warp=0.0  # warp is not valid for ultrasound geometry
    )
)

dls = fetal_block.dataloaders(FASTAI_DIR / "images", bs=8, num_workers=0)

# ✅ Sanity check — must print [0 1]
sample_x, sample_y = dls.train.dataset[0]
print("Mask unique values:", np.unique(np.array(sample_y)))  # expect [0 1]
print("Mask type:", type(sample_y))                          # expect FetalMask

dls.show_batch(max_n=4, vmin=0, vmax=1)

Training the U-Net Learner:

In [ ]:
from fastai.callback.tracker import SaveModelCallback

# Create U-Net learner with ResNet34 backbone
learn = unet_learner(
    dls, 
    resnet34, 
    loss_func=DiceFocalLoss(), 
    metrics=[DiceMulti()]
)

# 1. Find optimal learning rate with frozen encoder
suggested = learn.lr_find()
print(f"Suggested LR: {suggested.valley}")

# 2. Train frozen encoder for 30 epochs using the 1-cycle policy
learn.fit_one_cycle(30, lr_max=suggested.valley,
        cbs=[SaveModelCallback(monitor='dice_multi', comp=np.greater, fname='unet_best_frozen')]
)

# 3. Unfreeze the encoder for fine-tuning
learn.unfreeze()

# 4. Train unfrozen model for 20 epochs
learn.fit_one_cycle(20, lr_max=slice(1e-6, 1e-4),
    cbs=[SaveModelCallback(monitor='dice_multi', comp=np.greater, fname='unet_best_unfrozen')]
)

# Explicit save after full training run
learn.save('unet_resnet34_final')

Installing Dependencies:

In [ ]:
!pip install onnx
!pip install onnxruntime

Exporting FastAI U-Net to ONNX:

In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore")

# Load the best checkpoint
learn.load('unet_best_frozen')
# Put the model into evaluation mode
pytorch_model = learn.model.eval()

# Move model to CPU (ONNX export works more reliably on CPU)
pytorch_model = pytorch_model.cpu()

# FastAI UNet expects a 3-channel input due to ImageNet ResNet34 backbone, sized 256x256
dummy_input = torch.randn(1, 3, 256, 256)

# Export path
onnx_path = "/kaggle/working/unet-resnet34.onnx"

# Use legacy exporter (more compatible with FastAI models)
torch.onnx.export(
    pytorch_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,  # Older opset for better compatibility
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    verbose=False,
    # Force legacy exporter
    dynamo=False
)


# Validate the exported model
try:
    import onnx
    import onnxruntime as ort
    
    # Check ONNX model
    model_proto = onnx.load(onnx_path)
    onnx.checker.check_model(model_proto)
    print(f"✅ ONNX graph validated. Exported to {onnx_path}")
    
    # Optional: Test inference with ONNX Runtime
    ort_session = ort.InferenceSession(onnx_path)
    ort_inputs = {ort_session.get_inputs()[0].name: dummy_input.numpy()}
    ort_outputs = ort_session.run(None, ort_inputs)
    print("✅ ONNX Runtime inference successful!")
    
except ImportError:
    print("⚠️ ONNX Runtime not installed, skipping inference test")
except Exception as e:
    print(f"⚠️ Model validation warning: {e}")
    # Still consider export successful if basic validation passes